# STM32F407 DSP Pipeline — FIR Example (CubeMX → CubeIDE → Proteus)

This notebook builds an **ADC (PA1) → CMSIS-DSP FIR low-pass filter → DAC (PA4)** pipeline for the STM32F407, simulated in Proteus ISIS. It includes a TIM2 sample-rate calculator, CubeMX setup checklist, a code generator for the selected filter, Proteus wiring notes, expected behavior, and troubleshooting guidance.

---

### Scope & assumptions

- **MCU:** STM32F407VGT6.
- **Clock:** HSE 8 MHz → SYSCLK 168 MHz → APB1 timer clock 84 MHz. Use your board's actual clock source.
- **Filter:** 31-tap Hamming-windowed-sinc FIR low-pass; default cutoff 1 kHz at the actual TIM2 sample rate (default 10 kHz).
- **Processing:** ADC and DAC circular DMA buffers, processed in half/full-transfer callbacks.

> Proteus is useful for checking program flow and wiring, not analog precision or cycle-accurate performance.

## Part 0 — Setup

Import only the standard scientific stack. No STM32 tooling is required here—this notebook generates files you will paste into CubeIDE by hand.

In [ ]:
import os

print("Setup complete. Working directory:", os.getcwd())

## Part 1 — Timer Configuration Calculator

The sample rate is set by TIM2. Run the calculator below for the closest valid PSC/ARR values; it reports the actual sample rate used to design the filter. TIM2 has a 16-bit prescaler and a 32-bit auto-reload register.

In [1]:
import math

TIM2_CLOCK_HZ = 84_000_000  # APB1 timer clock on a 168 MHz F407

def compute_tim2(target_fs_hz, timer_clk_hz=TIM2_CLOCK_HZ):
    """Find the closest valid 16-bit PSC / 32-bit ARR pair for TIM2."""
    if not math.isfinite(target_fs_hz) or not math.isfinite(timer_clk_hz):
        raise ValueError("sample rate and timer clock must be finite")
    if target_fs_hz <= 0 or timer_clk_hz <= 0:
        raise ValueError("sample rate and timer clock must be positive")
    if target_fs_hz > timer_clk_hz:
        raise ValueError("target sample rate cannot exceed the TIM2 clock")
    min_fs_hz = timer_clk_hz / (65536 * (0xFFFFFFFF + 1))
    if target_fs_hz < min_fs_hz:
        raise ValueError(f"target sample rate is below the TIM2 minimum of {min_fs_hz:g} Hz")

    # Try PSC values from 0 upward; pick the one with smallest relative error.
    best = None
    for psc in range(0, 65536):
        denom = timer_clk_hz / (psc + 1)
        arr_f = denom / target_fs_hz - 1
        arr = int(round(arr_f))
        if arr < 0 or arr > 0xFFFFFFFF:
            continue
        actual = timer_clk_hz / ((psc + 1) * (arr + 1))
        err = abs(actual - target_fs_hz) / target_fs_hz
        if best is None or err < best[3]:
            best = (psc, arr, actual, err)
    if best is None:
        raise ValueError(f"No valid TIM2 PSC/ARR configuration for target sample rate {target_fs_hz} Hz")
    return best


# --- Set your desired sample rate here ---
TARGET_FS = 10_000  # Hz

psc, arr, actual_fs, err = compute_tim2(TARGET_FS)
print(f"Target sample rate : {TARGET_FS} Hz")
print(f"PSC (prescaler)    : {psc}")
print(f"ARR (counter per.) : {arr}")
print(f"Actual sample rate : {actual_fs:.3f} Hz  (error {err*100:.4f}%)")

if err < 0.01:
    print("\n✅ Good. Use these values in CubeMX. Design your filter for")
    print(f"   FS = {actual_fs:.1f} Hz.")
else:
    print("\n⚠️  Cannot hit target exactly with integer PSC/ARR.")
    print(f"   Design your filter for FS = {actual_fs:.1f} Hz.")

Target sample rate : 10000 Hz
PSC (prescaler)    : 0
ARR (counter per.) : 8399
Actual sample rate : 10000.000 Hz  (error 0.0000%)

✅ Good. Use these values in CubeMX. Design your filter for
   FS = 10000.0 Hz.


## Part 2 — CubeMX Configuration Checklist

Open **STM32CubeMX → New Project → STM32F407VGTx → Start Project**, then perform the following steps. Tick each box in the markdown cell below as you go (in Jupyter you can edit the markdown directly; in the browser you can just tick mentally).

### 2.1 System Core and Clock Tree
- [ ] **SYS → Debug**: **Serial Wire**.
- [ ] **RCC → HSE**: **Crystal/Ceramic Resonator** for an 8 MHz HSE crystal. Use the actual oscillator value/source for your board.
- [ ] Configure the clock tree for **SYSCLK/HCLK = 168 MHz**. For 8 MHz HSE, the usual PLL values are PLLM=8, PLLN=336, PLLP=2, PLLQ=7.
- [ ] Set **APB1 prescaler = /4** (PCLK1=42 MHz; TIM2 clock=84 MHz because the timer clock doubles when the APB prescaler is not 1). Set **APB2 prescaler = /2** (PCLK2=84 MHz).
- [ ] Confirm CubeMX shows **TIM2 clock = 84 MHz**; the Part 1 calculator depends on this. Keep CubeMX-generated flash latency and voltage settings for 168 MHz.
- [ ] **PA1**: `ADC1_IN1`, Analog mode, no pull. **PA4**: `DAC_OUT1`, Analog mode, no pull. Do not assign either pin to another peripheral.

### 2.2 ADC1 (PA1)
- [ ] Enable **IN1** (PA1)
- [ ] **Parameter Settings**:
  - [ ] Resolution **12 bits**; Data Alignment **Right**; Scan Conversion **Disabled**; Number Of Conversion **1**.
  - [ ] Continuous Conversion Mode **Disabled** and Discontinuous Conversion Mode **Disabled**. Each conversion must be started by the timer trigger.
  - [ ] External Trigger Conversion Source **Timer 2 Trigger Out event (TIM2_TRGO)**; trigger edge **Rising**.
  - [ ] DMA Continuous Requests **Enabled**; End Of Conversion Selection **End of single conversion**.
  - [ ] ADC clock prescaler **PCLK2 / 4** (21 MHz when PCLK2=84 MHz). Choose sampling time for the source impedance; **84 cycles** is a conservative starting point.
- [ ] **DMA Settings** → Add `ADC1`:
  - [ ] Direction **Peripheral to Memory**; Mode **Circular**; Memory increment **Enabled**, Peripheral increment **Disabled**.
  - [ ] Peripheral and Memory data width **Half Word**. Use a valid ADC1 mapping, e.g. **DMA2 Stream 0, Channel 0** (Stream 4, Channel 0 is also supported).
- [ ] Enable the selected ADC DMA stream interrupt in NVIC so the HAL half/full-complete callbacks are dispatched.

### 2.3 DAC (PA4)
- [ ] Enable **OUT1** (PA4)
- [ ] Configure DAC channel 1 trigger as **Timer 2 Trigger Out event (TIM2_TRGO)**; keep the output buffer enabled for typical loads.
- [ ] **DMA Settings** → Add `DAC1_CH1`:
  - [ ] Direction **Memory to Peripheral**; Mode **Circular**; Memory increment **Enabled**, Peripheral increment **Disabled**.
  - [ ] Peripheral and Memory data width **Half Word**. A common mapping is **DMA1 Stream 5, Channel 7**.
- [ ] Enable the selected DAC DMA stream interrupt in NVIC.

### 2.4 TIM2 (sampling clock)
- [ ] **Clock Source**: Internal Clock; counter direction **Up**; clock division **DIV1**.
- [ ] **Prescaler (PSC)** and **Counter Period (ARR)**: use Part 1 output. TIM2 has a 16-bit prescaler and 32-bit auto-reload register. For the default 10 kHz target at 84 MHz, use **PSC=0, ARR=8399** (`84,000,000 / ((0+1) × (8399+1)) = 10,000 Hz`).
- [ ] **Master Configuration → Trigger Output (TRGO)**: **Update Event**. This routes the timer update to the ADC and DAC trigger inputs; no PWM channel is needed.

### 2.5 Libraries
- [ ] Add CMSIS-DSP headers and a library built for Cortex-M4F using the project's floating-point ABI. Define `ARM_MATH_CM4` if it is not already set by the selected CMSIS package.

### 2.6 Project
- [ ] **Project Name**: `ADC_DSP_DAC`
- [ ] **Toolchain / IDE**: **STM32CubeIDE**
- [ ] Generate Code. Check that `MX_DMA_Init()` is called before ADC/DAC DMA is started and that DMA IRQ handlers call `HAL_DMA_IRQHandler()` for the selected streams.
- [ ] Confirm generated handles are `hadc1`, `hdac`, and `htim2` (or update the Part 3 snippet to match). Keep generated `SystemClock_Config()` and `MX_*_Init()` functions; add DSP code only at the indicated `USER CODE` markers.

---

> **Why a timer trigger and not software polling?** A hardware trigger gives a *jitter-free* sample interval, which is the single most important property for a clean digital filter. Polling or free-running ADC will smear the spectrum and make the filter appear to underperform.

## Part 3 — Generate the Filter-Specific `main.c` Snippet

Run this cell after Part 1 so the filter is designed for the **actual** timer sample rate. It writes one file with code for the selected filter; paste each marked section into the matching CubeIDE `USER CODE` block. Do not overwrite CubeIDE's generated `main.c` or its initialization functions.

The FIR and IIR notebooks are alternatives: use only one filter's generated callbacks in a project. The snippet initializes the DAC buffer to midscale, starts ADC/DAC DMA before TIM2, and checks HAL start errors. Each completed ADC half is written to the same-index DAC half, which DMA has already consumed; this avoids racing the DAC DMA.

In [ ]:
import math

BLOCK_SIZE = 32
FS_HZ = float(actual_fs)
CUTOFF_HZ = 1000.0
NUM_TAPS = 31  # Odd length gives an integer linear-phase group delay.

if BLOCK_SIZE < 1:
    raise ValueError("BLOCK_SIZE must be positive")
if NUM_TAPS < 3 or NUM_TAPS % 2 == 0:
    raise ValueError("NUM_TAPS must be an odd integer of at least 3")
if not 0.0 < CUTOFF_HZ < FS_HZ / 2.0:
    raise ValueError("CUTOFF_HZ must be between 0 and the Nyquist frequency")

# Windowed-sinc low-pass, normalized for unity DC gain.
mid = (NUM_TAPS - 1) / 2.0
fir_coeffs = []
for n in range(NUM_TAPS):
    offset = n - mid
    ideal = 2.0 * CUTOFF_HZ / FS_HZ if offset == 0 else math.sin(2.0 * math.pi * CUTOFF_HZ * offset / FS_HZ) / (math.pi * offset)
    window = 0.54 - 0.46 * math.cos(2.0 * math.pi * n / (NUM_TAPS - 1))
    fir_coeffs.append(ideal * window)
coefficient_sum = sum(fir_coeffs)
fir_coeffs = [coefficient / coefficient_sum for coefficient in fir_coeffs]
coeff_text = ",\n    ".join(f"{coefficient:.10e}f" for coefficient in fir_coeffs)

MAIN_C_TEMPLATE = r'''/* USER CODE BEGIN Includes */
#include "arm_math.h"
/* USER CODE END Includes */

/* USER CODE BEGIN PV */
#define BLOCK_SIZE       @BLOCK_SIZE@u
#define FULL_BUFFER_SIZE (2u * BLOCK_SIZE)
#define NUM_TAPS         @NUM_TAPS@u

static uint16_t adc_buffer[FULL_BUFFER_SIZE];
static uint16_t dac_buffer[FULL_BUFFER_SIZE];
static float32_t input_f32[BLOCK_SIZE];
static float32_t output_f32[BLOCK_SIZE];
static const float32_t fir_coeffs[NUM_TAPS] = {
    @COEFFICIENTS@
};
static float32_t fir_state[NUM_TAPS + BLOCK_SIZE - 1u];
static arm_fir_instance_f32 S_FIR;
static volatile uint16_t *adc_in_ptr;
static volatile uint16_t *dac_out_ptr;
static void process_half_buffer(void);
/* USER CODE END PV */

/* USER CODE BEGIN 2 */
for (uint32_t i = 0; i < FULL_BUFFER_SIZE; ++i) dac_buffer[i] = 2048u;
arm_fir_init_f32(&S_FIR, NUM_TAPS, fir_coeffs, fir_state, BLOCK_SIZE);
if (HAL_ADC_Start_DMA(&hadc1, (uint32_t *)adc_buffer, FULL_BUFFER_SIZE) != HAL_OK) Error_Handler();
if (HAL_DAC_Start_DMA(&hdac, DAC_CHANNEL_1, (uint32_t *)dac_buffer, FULL_BUFFER_SIZE, DAC_ALIGN_12B_R) != HAL_OK) Error_Handler();
if (HAL_TIM_Base_Start(&htim2) != HAL_OK) Error_Handler();
/* USER CODE END 2 */

/* USER CODE BEGIN 4 */
void HAL_ADC_ConvHalfCpltCallback(ADC_HandleTypeDef *hadc)
{
    if (hadc != &hadc1) return;
    adc_in_ptr = &adc_buffer[0];
    dac_out_ptr = &dac_buffer[0];
    process_half_buffer();
}

void HAL_ADC_ConvCpltCallback(ADC_HandleTypeDef *hadc)
{
    if (hadc != &hadc1) return;
    adc_in_ptr = &adc_buffer[BLOCK_SIZE];
    dac_out_ptr = &dac_buffer[BLOCK_SIZE];
    process_half_buffer();
}

static void process_half_buffer(void)
{
    for (uint32_t i = 0; i < BLOCK_SIZE; ++i) {
        input_f32[i] = ((float32_t)adc_in_ptr[i] - 2048.0f) / 2048.0f;
    }

    arm_fir_f32(&S_FIR, input_f32, output_f32, BLOCK_SIZE);

    for (uint32_t i = 0; i < BLOCK_SIZE; ++i) {
        float32_t sample = output_f32[i] * 2048.0f + 2048.0f;
        if (sample > 4095.0f) sample = 4095.0f;
        if (sample < 0.0f) sample = 0.0f;
        dac_out_ptr[i] = (uint16_t)sample;
    }
}
/* USER CODE END 4 */
'''
MAIN_C_TEMPLATE = MAIN_C_TEMPLATE.replace("@COEFFICIENTS@", coeff_text)
MAIN_C_TEMPLATE = MAIN_C_TEMPLATE.replace("@BLOCK_SIZE@", str(BLOCK_SIZE))
MAIN_C_TEMPLATE = MAIN_C_TEMPLATE.replace("@NUM_TAPS@", str(NUM_TAPS))
out_path = "main_user_code_snippets_fir.c"
with open(out_path, "w", encoding="utf-8") as output_file:
    output_file.write(MAIN_C_TEMPLATE)

print(f"FIR: {NUM_TAPS} taps, cutoff {CUTOFF_HZ:g} Hz, sample rate {FS_HZ:g} Hz")
print("FIR coefficients:", [f"{coefficient:.10e}" for coefficient in fir_coeffs])
print(f"Wrote {out_path} ({len(MAIN_C_TEMPLATE)} characters)")

## Part 4 — Build the `.hex` File in CubeIDE

1. In the **Project Explorer**, right-click your project → **Properties**.
2. **C/C++ Build → Settings → Tool Settings → MCU/MPU Post build outputs**.
3. ✅ Tick **Convert to Intel Hex file**.
4. **Apply and Close** → **Project → Build Project** (or click the hammer).
5. The `.hex` file appears at:
   `Debug/ADC_DSP_DAC.hex` (relative to your project root).

**Quick verification — run this cell after you have built:**

If you have STM32CubeIDE installed locally and built the project, the cell below will *try* to locate the `.hex` file for you. Otherwise it will just print the expected path.

In [ ]:
import glob, os

hex_candidates = glob.glob("./**/Debug/*.hex", recursive=True) + \
                 glob.glob("./**/Release/*.hex", recursive=True)

if hex_candidates:
    print("Found .hex file(s):")
    for p in hex_candidates:
        print(f"  {os.path.abspath(p)}  ({os.path.getsize(p)} bytes)")
else:
    print("No .hex found in this working directory.")
    print("Expected location: <project>/Debug/ADC_DSP_DAC.hex")
    print("Make sure you enabled 'Convert to Intel Hex file' in project properties.")

## Part 5 — Proteus ISIS Circuit Checklist

### 5.1 STM32F407 Model Availability

The plain `STM32F407` is not always present in Proteus. Search for one of:

- **`STM32F407VET6`** — most commonly available third-party model
- **`STM32F407VGT6`** — same family, if you find it

If neither appears in the **Pick Devices** dialog, you must install a third-party model library (`.LIB` + `.DLL`) into Proteus' `LIBRARY` folder. This is a common requirement — many public STM32 model packs exist for this exact reason.

### 5.2 Component List

| Designator | Search keyword | Notes |
|:---|:---|:---|
| U1 | `STM32F407VxT6` | The MCU |
| V1 | `VSINE` | Sine generator → drives PA1 |
| R1 (optional) | `RES` 1kΩ | Series protection into PA1 |
| — | `OSCILLOSCOPE` | A-channel on **PA4**, B-channel on **PA1** |
| — | `POWER` / `GROUND` | Tie to VDD / VSS pins of MCU |

### 5.3 MCU Properties (double-click U1)

- **Program File**: point to the `.hex` file from Part 4.
- **Crystal Frequency**: **8 MHz** (matches your HSE config in CubeMX).
- **Clock Frequency**: leave as **default**; STM32CubeMX's own clock init will configure the PLL at startup.

### 5.4 Wiring

- **VDD (pins 11, 19, 32, 48, 64, 75, 100)** → **VDD/VCC**
- **VSS (pins 12, 18, 33, 47, 63, 74, 99)** → **GROUND**
- **VSINE** output → **PA1** (pin 23 on the LQFP100 package)
- **PA4** (pin 28) → **Oscilloscope channel A**
- **PA1** → **Oscilloscope channel B** (for side-by-side comparison)

> **Common Proteus gotcha:** forgetting to power the VDD pins. If the MCU appears inert (no clock, no output), this is almost always the cause.

## Part 6 — Expected Behaviour (FIR)

The FIR notebook uses a 31-tap Hamming-windowed-sinc low-pass, normalized to unity DC gain. At the default 10 kHz sample rate and 1 kHz cutoff, low-frequency inputs should pass and components well above the cutoff should be attenuated.

- **DC input:** settles to the same level after the FIR's finite startup transient.
- **Step input:** the output transitions over the filter's tap span.
- **Filter group delay:** `(NUM_TAPS - 1) / 2 = 15` samples (1.5 ms at 10 kHz).
- **End-to-end delay:** writing into the same DAC half only after DMA has consumed it adds about `2 × BLOCK_SIZE` samples (6.4 ms at 10 kHz), in addition to filter delay.

Regenerate coefficients if sample rate or cutoff changes. Increasing tap count narrows the transition band but increases computation and filter delay.

## Part 7 — FIR Troubleshooting

| Symptom | Likely cause | Check |
|:---|:---|:---|
| MCU does not run | Proteus VDD/VSS pins are unpowered | Wire all supply and ground pins |
| No DAC output | DMA, timer, or trigger setup is wrong | Check CubeMX trigger selections and HAL start return values |
| Glitches or stale blocks | Processing missed its half-buffer deadline | Measure callback time; ensure `FULL_BUFFER_SIZE = 2 * BLOCK_SIZE` |
| Little/no filtering | Coefficients or cutoff are not what you expect | Inspect generated `fir_coeffs[]`; rerun after changing FS/cutoff |
| Wrong cutoff | Timer rate differs from coefficient design rate | Re-run Part 1, update CubeMX, and regenerate the snippet |
| High-frequency input aliases | No analog anti-alias filtering before ADC | Add a suitable analog low-pass ahead of PA1; digital filtering cannot remove already-aliased energy |

## Part 8 — Going Further

- Compare this FIR example with [the IIR example](./stm32f407vg_iir_example.ipynb). The FIR has linear phase and fixed group delay; the second-order IIR has frequency-dependent phase.
- The default FIR uses 31 taps and a 1 kHz cutoff. Change `NUM_TAPS` or `CUTOFF_HZ` in Part 3 and rerun it to generate new coefficients. Keep `NUM_TAPS` odd.
- The default timer is 10 kHz (`PSC=0`, `ARR=8399`). To increase the rate to 100 kHz, reduce `ARR` by about 10× to 839, then rerun the coefficient-generation cell. For other rates, use Part 1's calculator.
- Increase `BLOCK_SIZE` only after measuring callback time and end-to-end latency; a larger block increases buffering latency even when CPU utilization remains acceptable.
- Add an analog RC anti-alias low-pass before PA1. Digital filtering cannot remove frequencies that have already aliased during sampling.

**Generated artifact:** `main_user_code_snippets_fir.c` contains the FIR-specific CubeIDE `USER CODE` sections.